<a href="https://colab.research.google.com/github/sergezolo/Active-Record-Association-Methods-onl01-seng-pt-032320/blob/master/Individual_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
FoodHub - Exploratory Data Analysis (EDA)
ISTM 650: AI Strategy for Business Decision Making — Capstone Checkpoint 1

This script performs:
1. Data overview and quality assessment
2. Univariate and bivariate exploratory analysis
3. Visualization dashboard
4. Key business insight generation

Author: [Your Name]
Date: [Current Date]
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SECTION 1: DATA LOADING AND OVERVIEW
# ============================================================

print("=" * 70)
print("FOODHUB — EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# Load the dataset
df = pd.read_csv('foodhub_order.csv')

print(f"\n✓ Dataset loaded successfully")
print(f"  Total orders : {len(df):,}")
print(f"  Variables    : {len(df.columns)}")

print("\n--- First 5 Records ---")
print(df.head())

print("\n--- Dataset Info ---")
print(df.info())

print("\n--- Descriptive Statistics (Numeric) ---")
print(df.describe().round(2))

print("\n--- Descriptive Statistics (Categorical) ---")
print(df.describe(include='object'))

# ============================================================
# SECTION 2: DATA QUALITY ASSESSMENT
# ============================================================

print("\n" + "=" * 70)
print("SECTION 2: DATA QUALITY ASSESSMENT")
print("=" * 70)

# 2a. Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# 2b. Duplicates
dup_count = df.duplicated().sum()
print(f"\nDuplicate rows: {dup_count}")

# 2c. Rating column — "Not given" treated as missing
rating_not_given = (df['rating'] == 'Not given').sum()
rating_given = len(df) - rating_not_given
print(f"\nRatings provided : {rating_given} ({rating_given/len(df)*100:.1f}%)")
print(f"Ratings not given: {rating_not_given} ({rating_not_given/len(df)*100:.1f}%)")

# Convert rating to numeric for analysis (Not given → NaN)
df['rating_numeric'] = pd.to_numeric(df['rating'], errors='coerce')

# 2d. Unique counts
print(f"\nUnique customers    : {df['customer_id'].nunique()}")
print(f"Unique restaurants  : {df['restaurant_name'].nunique()}")
print(f"Unique cuisine types: {df['cuisine_type'].nunique()}")
print(f"Cuisine types       : {df['cuisine_type'].unique()}")

# 2e. Outlier check — cost
print("\n--- Cost of Order Distribution ---")
print(df['cost_of_the_order'].describe().round(2))
q1 = df['cost_of_the_order'].quantile(0.25)
q3 = df['cost_of_the_order'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
cost_outliers = ((df['cost_of_the_order'] < lower_bound) |
                 (df['cost_of_the_order'] > upper_bound)).sum()
print(f"IQR cost outliers (1.5×IQR): {cost_outliers}")

# 2f. Outlier check — food preparation time
print("\n--- Food Preparation Time Distribution ---")
print(df['food_preparation_time'].describe().round(2))

# 2g. Outlier check — delivery time
print("\n--- Delivery Time Distribution ---")
print(df['delivery_time'].describe().round(2))

# ============================================================
# SECTION 3: FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 70)
print("SECTION 3: FEATURE ENGINEERING")
print("=" * 70)

# Total time from order to delivery
df['total_time'] = df['food_preparation_time'] + df['delivery_time']

# Cost buckets
df['cost_bucket'] = pd.cut(
    df['cost_of_the_order'],
    bins=[0, 10, 20, 30, np.inf],
    labels=['Low ($0-10)', 'Medium ($10-20)', 'High ($20-30)', 'Premium ($30+)']
)

# Flag: high cost order (top 25%)
cost_75 = df['cost_of_the_order'].quantile(0.75)
df['high_value_order'] = (df['cost_of_the_order'] >= cost_75).astype(int)

# Flag: rated order
df['is_rated'] = (df['rating'] != 'Not given').astype(int)

print("✓ Created: total_time, cost_bucket, high_value_order, is_rated")

# ============================================================
# SECTION 4: KEY FINDINGS & ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("SECTION 4: KEY FINDINGS")
print("=" * 70)

# ----- FINDING 1: Cuisine Type Revenue & Volume -----
print("\n--- Finding 1: Revenue & Volume by Cuisine Type ---")
cuisine_summary = df.groupby('cuisine_type').agg(
    order_count=('order_id', 'count'),
    total_revenue=('cost_of_the_order', 'sum'),
    avg_order_value=('cost_of_the_order', 'mean'),
    avg_prep_time=('food_preparation_time', 'mean'),
    avg_delivery_time=('delivery_time', 'mean'),
    avg_rating=('rating_numeric', 'mean')
).round(2)
cuisine_summary['revenue_pct'] = (
    cuisine_summary['total_revenue'] /
    cuisine_summary['total_revenue'].sum() * 100
).round(1)
cuisine_summary = cuisine_summary.sort_values('total_revenue', ascending=False)
print(cuisine_summary)

# ----- FINDING 2: Top 15 Restaurants by Order Volume -----
print("\n--- Finding 2: Top 15 Restaurants by Order Count ---")
top_restaurants = df.groupby('restaurant_name').agg(
    order_count=('order_id', 'count'),
    total_revenue=('cost_of_the_order', 'sum'),
    avg_order_value=('cost_of_the_order', 'mean'),
    avg_rating=('rating_numeric', 'mean')
).round(2).sort_values('order_count', ascending=False).head(15)
print(top_restaurants)

# ----- FINDING 3: Weekend vs Weekday -----
print("\n--- Finding 3: Weekday vs Weekend Patterns ---")
day_summary = df.groupby('day_of_the_week').agg(
    order_count=('order_id', 'count'),
    avg_cost=('cost_of_the_order', 'mean'),
    avg_prep_time=('food_preparation_time', 'mean'),
    avg_delivery_time=('delivery_time', 'mean'),
    avg_total_time=('total_time', 'mean'),
    avg_rating=('rating_numeric', 'mean'),
    pct_rated=('is_rated', 'mean')
).round(2)
print(day_summary)

# ----- FINDING 4: Rating Analysis -----
print("\n--- Finding 4: Rating Distribution (rated orders only) ---")
rated_df = df[df['rating_numeric'].notna()]
print(rated_df['rating_numeric'].value_counts().sort_index())
print(f"\nMean rating: {rated_df['rating_numeric'].mean():.2f}")
print(f"Median rating: {rated_df['rating_numeric'].median():.1f}")

# Rating by cuisine
print("\nAverage Rating by Cuisine (rated orders):")
print(rated_df.groupby('cuisine_type')['rating_numeric'].agg(
    ['mean', 'count']).round(2).sort_values('mean', ascending=False))

# ----- FINDING 5: Operational Efficiency -----
print("\n--- Finding 5: Preparation & Delivery Time Analysis ---")
print(f"Avg food prep time  : {df['food_preparation_time'].mean():.1f} min")
print(f"Avg delivery time   : {df['delivery_time'].mean():.1f} min")
print(f"Avg total time      : {df['total_time'].mean():.1f} min")
print(f"Median total time   : {df['total_time'].median():.1f} min")

# Long-wait orders (total_time > 60 min)
long_wait = (df['total_time'] > 60).sum()
print(f"\nOrders > 60 min total: {long_wait} ({long_wait/len(df)*100:.1f}%)")

# Correlation: does longer time correlate with lower rating?
print("\nCorrelation matrix (numeric variables, rated orders):")
corr_cols = ['cost_of_the_order', 'food_preparation_time',
             'delivery_time', 'total_time', 'rating_numeric']
print(rated_df[corr_cols].corr().round(3))

# ----- FINDING 6: Customer Ordering Behavior -----
print("\n--- Finding 6: Customer Order Frequency ---")
cust_orders = df.groupby('customer_id').agg(
    num_orders=('order_id', 'count'),
    total_spent=('cost_of_the_order', 'sum'),
    avg_spent=('cost_of_the_order', 'mean')
).round(2)
print(cust_orders['num_orders'].describe().round(2))
repeat_customers = (cust_orders['num_orders'] > 1).sum()
single_customers = (cust_orders['num_orders'] == 1).sum()
print(f"\nSingle-order customers: {single_customers} "
      f"({single_customers/len(cust_orders)*100:.1f}%)")
print(f"Repeat customers      : {repeat_customers} "
      f"({repeat_customers/len(cust_orders)*100:.1f}%)")

# Top repeat customers
print("\nTop 10 customers by order count:")
print(cust_orders.sort_values('num_orders', ascending=False).head(10))

# ----- FINDING 7: Cost vs Rating -----
print("\n--- Finding 7: Cost Bucket vs Rating ---")
cost_rating = rated_df.groupby('cost_bucket').agg(
    avg_rating=('rating_numeric', 'mean'),
    count=('rating_numeric', 'count')
).round(2)
print(cost_rating)

# ============================================================
# SECTION 5: VISUALIZATIONS
# ============================================================

print("\n" + "=" * 70)
print("SECTION 5: CREATING VISUALIZATIONS")
print("=" * 70)

sns.set_style("whitegrid")
palette = sns.color_palette("Set2")

fig = plt.figure(figsize=(22, 28))

# --- Plot 1: Orders by Cuisine Type ---
ax1 = plt.subplot(4, 3, 1)
cuisine_counts = df['cuisine_type'].value_counts()
cuisine_counts.plot(kind='bar', color=palette, ax=ax1, edgecolor='black')
ax1.set_title('Order Volume by Cuisine Type', fontsize=13, fontweight='bold')
ax1.set_xlabel('Cuisine Type')
ax1.set_ylabel('Number of Orders')
plt.xticks(rotation=45, ha='right')

# --- Plot 2: Revenue by Cuisine Type ---
ax2 = plt.subplot(4, 3, 2)
cuisine_rev = df.groupby('cuisine_type')['cost_of_the_order'].sum().sort_values(ascending=False)
cuisine_rev.plot(kind='bar', color='steelblue', ax=ax2, edgecolor='black')
ax2.set_title('Total Revenue by Cuisine Type', fontsize=13, fontweight='bold')
ax2.set_xlabel('Cuisine Type')
ax2.set_ylabel('Total Revenue ($)')
plt.xticks(rotation=45, ha='right')

# --- Plot 3: Top 10 Restaurants by Orders ---
ax3 = plt.subplot(4, 3, 3)
top10_rest = df['restaurant_name'].value_counts().head(10)
top10_rest.plot(kind='barh', color='coral', ax=ax3, edgecolor='black')
ax3.set_title('Top 10 Restaurants by Order Count', fontsize=13, fontweight='bold')
ax3.set_xlabel('Number of Orders')
ax3.invert_yaxis()

# --- Plot 4: Cost Distribution ---
ax4 = plt.subplot(4, 3, 4)
df['cost_of_the_order'].hist(bins=30, color='teal', alpha=0.7, ax=ax4,
                              edgecolor='black')
ax4.axvline(df['cost_of_the_order'].mean(), color='red', linestyle='--',
            linewidth=2, label=f"Mean: ${df['cost_of_the_order'].mean():.2f}")
ax4.axvline(df['cost_of_the_order'].median(), color='orange', linestyle='--',
            linewidth=2, label=f"Median: ${df['cost_of_the_order'].median():.2f}")
ax4.set_title('Distribution of Order Cost', fontsize=13, fontweight='bold')
ax4.set_xlabel('Cost ($)')
ax4.set_ylabel('Frequency')
ax4.legend()

# --- Plot 5: Rating Distribution ---
ax5 = plt.subplot(4, 3, 5)
rated_df['rating_numeric'].value_counts().sort_index().plot(
    kind='bar', color='mediumpurple', ax=ax5, edgecolor='black')
ax5.set_title('Rating Distribution (Rated Orders)', fontsize=13, fontweight='bold')
ax5.set_xlabel('Rating')
ax5.set_ylabel('Count')
plt.xticks(rotation=0)

# --- Plot 6: Weekday vs Weekend ---
ax6 = plt.subplot(4, 3, 6)
day_counts = df['day_of_the_week'].value_counts()
day_counts.plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax6, edgecolor='black')
ax6.set_title('Orders: Weekday vs Weekend', fontsize=13, fontweight='bold')
ax6.set_xlabel('Day Type')
ax6.set_ylabel('Order Count')
plt.xticks(rotation=0)

# --- Plot 7: Preparation Time Distribution ---
ax7 = plt.subplot(4, 3, 7)
df['food_preparation_time'].hist(bins=20, color='goldenrod', alpha=0.7,
                                  ax=ax7, edgecolor='black')
ax7.axvline(df['food_preparation_time'].mean(), color='red', linestyle='--',
            label=f"Mean: {df['food_preparation_time'].mean():.1f} min")
ax7.set_title('Food Preparation Time Distribution', fontsize=13, fontweight='bold')
ax7.set_xlabel('Minutes')
ax7.set_ylabel('Frequency')
ax7.legend()

# --- Plot 8: Delivery Time Distribution ---
ax8 = plt.subplot(4, 3, 8)
df['delivery_time'].hist(bins=20, color='lightseagreen', alpha=0.7,
                          ax=ax8, edgecolor='black')
ax8.axvline(df['delivery_time'].mean(), color='red', linestyle='--',
            label=f"Mean: {df['delivery_time'].mean():.1f} min")
ax8.set_title('Delivery Time Distribution', fontsize=13, fontweight='bold')
ax8.set_xlabel('Minutes')
ax8.set_ylabel('Frequency')
ax8.legend()

# --- Plot 9: Avg Order Value by Cuisine ---
ax9 = plt.subplot(4, 3, 9)
avg_cost_cuisine = df.groupby('cuisine_type')['cost_of_the_order'].mean().sort_values(ascending=False)
avg_cost_cuisine.plot(kind='bar', color='salmon', ax=ax9, edgecolor='black')
ax9.set_title('Average Order Value by Cuisine', fontsize=13, fontweight='bold')
ax9.set_xlabel('Cuisine Type')
ax9.set_ylabel('Avg Cost ($)')
plt.xticks(rotation=45, ha='right')

# --- Plot 10: Heatmap — Correlation ---
ax10 = plt.subplot(4, 3, 10)
corr_matrix = df[['cost_of_the_order', 'food_preparation_time',
                   'delivery_time', 'total_time']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            ax=ax10, linewidths=0.5)
ax10.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')

# --- Plot 11: Box Plot — Cost by Cuisine ---
ax11 = plt.subplot(4, 3, 11)
order_by_median = df.groupby('cuisine_type')['cost_of_the_order'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='cuisine_type', y='cost_of_the_order',
            order=order_by_median, palette='Set2', ax=ax11)
ax11.set_title('Order Cost by Cuisine Type', fontsize=13, fontweight='bold')
ax11.set_xlabel('Cuisine Type')
ax11.set_ylabel('Cost ($)')
plt.xticks(rotation=45, ha='right')

# --- Plot 12: Total Time vs Rating (rated only) ---
ax12 = plt.subplot(4, 3, 12)
if len(rated_df) > 0:
    sns.boxplot(data=rated_df, x='rating_numeric', y='total_time',
                palette='viridis', ax=ax12)
    ax12.set_title('Total Time by Rating', fontsize=13, fontweight='bold')
    ax12.set_xlabel('Rating')
    ax12.set_ylabel('Total Time (min)')

plt.tight_layout(pad=3.0)
plt.savefig('FoodHub_EDA_Dashboard.png', dpi=300, bbox_inches='tight')
print("\n✓ Dashboard saved as 'FoodHub_EDA_Dashboard.png'")
plt.show()

# ============================================================
# ADDITIONAL VISUALIZATION: Cuisine Revenue Share (Pie)
# ============================================================

fig2, ax = plt.subplots(figsize=(8, 8))
cuisine_rev_share = df.groupby('cuisine_type')['cost_of_the_order'].sum().sort_values(ascending=False)
ax.pie(cuisine_rev_share, labels=cuisine_rev_share.index,
       autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
ax.set_title('Revenue Share by Cuisine Type', fontsize=14, fontweight='bold')
plt.savefig('FoodHub_Revenue_Pie.png', dpi=300, bbox_inches='tight')
print("✓ Revenue pie chart saved as 'FoodHub_Revenue_Pie.png'")
plt.show()

# ============================================================
# SECTION 6: SUMMARY STATISTICS EXPORT
# ============================================================

print("\n" + "=" * 70)
print("SECTION 6: SUMMARY")
print("=" * 70)

summary = pd.DataFrame({
    'Metric': [
        'Total Orders',
        'Unique Customers',
        'Unique Restaurants',
        'Unique Cuisines',
        'Total Revenue',
        'Average Order Value',
        'Median Order Value',
        'Avg Food Prep Time (min)',
        'Avg Delivery Time (min)',
        'Avg Total Time (min)',
        'Rating Coverage (%)',
        'Mean Rating (rated only)',
        'Weekend Order %',
        'Repeat Customer %'
    ],
    'Value': [
        len(df),
        df['customer_id'].nunique(),
        df['restaurant_name'].nunique(),
        df['cuisine_type'].nunique(),
        f"${df['cost_of_the_order'].sum():,.2f}",
        f"${df['cost_of_the_order'].mean():.2f}",
        f"${df['cost_of_the_order'].median():.2f}",
        f"{df['food_preparation_time'].mean():.1f}",
        f"{df['delivery_time'].mean():.1f}",
        f"{df['total_time'].mean():.1f}",
        f"{rating_given/len(df)*100:.1f}%",
        f"{rated_df['rating_numeric'].mean():.2f}",
        f"{(df['day_of_the_week']=='Weekend').mean()*100:.1f}%",
        f"{repeat_customers/len(cust_orders)*100:.1f}%"
    ]
})
print(summary.to_string(index=False))

summary.to_csv('FoodHub_Summary_Statistics.csv', index=False)
print("\n✓ Summary saved to FoodHub_Summary_Statistics.csv")

# ============================================================
# SECTION 7: BUSINESS QUESTIONS FOR FURTHER INVESTIGATION
# ============================================================

print("\n" + "=" * 70)
print("SECTION 7: BUSINESS QUESTIONS FOR FURTHER INVESTIGATION")
print("=" * 70)

questions = [
    "1. What drives the ~39% of customers who do NOT rate their orders, "
    "and can we predict or incentivize rating behavior?",
    "2. Which customer segments (based on order frequency, spend, and "
    "cuisine preference) are most valuable, and how should FoodHub "
    "personalize their experience?",
    "3. Do longer food preparation or delivery times causally reduce "
    "ratings, and what operational thresholds trigger dissatisfaction?",
    "4. Which restaurant–cuisine combinations are under-represented "
    "relative to demand, representing partnership expansion opportunities?",
    "5. Can we build a recommendation engine using customer ordering "
    "history to predict cuisine/restaurant preferences and increase "
    "order frequency?"
]
for q in questions:
    print(f"\n  {q}")

print("\n" + "=" * 70)
print("EDA COMPLETE")
print("=" * 70)